In [1]:
from serpapi import GoogleSearch
import pandas as pd
import re
import time
from tqdm import tqdm
import requests
from bs4 import BeautifulSoup
import numpy as np  

# -------------------------------
# CONFIG
# -------------------------------
SERPAPI_KEY = "a4de96442600cb70c849304d37271b41342083f5213509ad2f349bb0d57de778"

# -------------------------------
# LOAD CSV
# -------------------------------
df = pd.read_csv("final_makati_activities.csv")

# -------------------------------
# FILTER NON-TICKET PLACES
# -------------------------------
invalid_categories = [
    "park", "mall", "shopping", "store",
    "hotel", "bar", "restaurant", "cafe"
]

# -------------------------------
# PRICE EXTRACTION ENGINE
# -------------------------------
def extract_price_range(text):
    if not text:
        return None

    text_lower = text.lower()

    patterns = [
        r'(?:₱|php|p)\s?\d{1,3}(?:,\d{3})*(?:\.\d{1,2})?',
        r'\d{1,3}(?:,\d{3})*(?:\.\d{1,2})?\s?(?:php|pesos?)',
        r'(?:₱|php|p)\s?\d+\s?[-–]\s?(?:₱|php|p)?\s?\d+',
        r'\d+\s?[-–]\s?\d+\s?(?:php|pesos?)',
        r'\d+(?:\.\d{1,2})?\s?per\s?(?:person|pax|head|visitor)',
        r'\d+(?:\.\d{1,2})?\s?per\s?(?:hour|hr|session)',
        r'\d+(?:\.\d{1,2})?\s?per\s?(?:game|ride|entry)',
        r'(?:ticket|admission|entrance|entry)\s?(?:price|fee)?[:\s]*\d+',
        r'(?:adult|child|student|senior)\s?(?:price)?[:\s]*\d+',
        r'\d+\s?(?:for|/)\s?\d+\s?(?:games|rides|hours)',
        r'\d+\s?(?:credits?|tokens?)',
        r'(?:price|cost|rate|fee)\s?(?:is|:)?\s?\d+'
    ]

    prices = []

    for pattern in patterns:
        matches = re.findall(pattern, text_lower)
        for m in matches:
            num = re.sub(r'[^\d.]', '', m)
            if num:
                try:
                    prices.append(int(float(num)))
                except:
                    continue

    prices = [p for p in prices if 50 <= p <= 5000]

    if not prices:
        if "free" in text_lower:
            return "FREE"
        return None

    if "per hour" in text_lower:
        unit = "per_hour"
    elif "per game" in text_lower:
        unit = "per_game"
    elif "per ride" in text_lower:
        unit = "per_ride"
    elif "per person" in text_lower or "pax" in text_lower:
        unit = "per_person"
    else:
        unit = "per_ticket"

    min_price = min(prices)
    max_price = max(prices)

    if min_price == max_price:
        return f"{min_price} ({unit})"

    return f"{min_price}-{max_price} ({unit})"


# -------------------------------
# SCRAPE PRICE FROM WEBPAGE
# -------------------------------
def extract_price_from_page(url):
    try:
        headers = {"User-Agent": "Mozilla/5.0"}
        response = requests.get(url, headers=headers, timeout=6)

        if response.status_code != 200:
            return None

        soup = BeautifulSoup(response.text, "html.parser")
        text = soup.get_text(" ", strip=True)

        return extract_price_range(text)

    except:
        return None


# -------------------------------
# GOOGLE SEARCH SCRAPER
# -------------------------------
def get_price_range(name):

    queries = [

        # 🎯 CORE
        f"{name} price Philippines",
        f"{name} cost Philippines",
        f"{name} rates Philippines",

        # 🎟️ ENTRY / TICKETS
        f"{name} ticket price Philippines",
        f"{name} entrance fee Philippines",
        f"{name} admission fee Philippines",
        f"{name} entry fee Philippines",
        f"{name} entry price Philippines",

        # 👤 PER PERSON
        f"{name} price per person",
        f"{name} cost per person",
        f"{name} rate per person",

        # 👨‍👩‍👧‍👦 CATEGORY PRICING
        f"{name} adult ticket price",
        f"{name} student price",
        f"{name} child price",
        f"{name} kids price",

        # 🎮 ACTIVITY-SPECIFIC
        f"{name} rates Makati",
        f"{name} pricing Makati",
        f"{name} packages Makati",
        f"{name} credits price",
        f"{name} tokens price",

        # ⏱️ TIME-BASED
        f"{name} per hour rate",
        f"{name} hourly rate",
        f"{name} session price",

        # 🎭 EVENTS
        f"{name} show ticket price",
        f"{name} event ticket price",

        # 🌐 PLATFORMS
        f"{name} Klook price",
        f"{name} TripAdvisor price",
        f"{name} official website price",

        # 💡 FALLBACK
        f"{name} Philippines how much",
        f"{name} Makati how much",
        f"{name} cost Makati"
    ]

    for q in queries:
        params = {
            "engine": "google",
            "q": q,
            "hl": "en",
            "api_key": SERPAPI_KEY
        }

        search = GoogleSearch(params)
        results = search.get_dict()

        for item in results.get("organic_results", []):
            snippet = item.get("snippet", "")
            link = item.get("link")

            price = extract_price_range(snippet)

            if not price and link:
                print(f"🔗 Checking: {link}")
                price = extract_price_from_page(link)

            if price:
                return price

        time.sleep(0.3)

    return None


# -------------------------------
# CLEAN FORMAT
# -------------------------------
def clean_price(val):
    val = str(val).lower()

    if "free" in val:
        return 0, 0, "free"

    if "not_ticket_based" in val:
        return np.nan, np.nan, "none"

    nums = re.findall(r'\d+', val)
    nums = [int(n) for n in nums]

    if not nums:
        return np.nan, np.nan, "unknown"

    if "person" in val:
        unit = "person"
    elif "hour" in val:
        unit = "hour"
    elif "game" in val:
        unit = "game"
    else:
        unit = "ticket"

    if len(nums) == 1:
        return nums[0], nums[0], unit
    else:
        return min(nums), max(nums), unit


# -------------------------------
# APPLY SCRAPER
# -------------------------------
price_ranges = []

print("🔄 STARTING SMART PRICE SCRAPE...\n")

for i, row in tqdm(df.iterrows(), total=len(df), desc="Scraping Prices"):

    name = row["activity_name"]  # ✅ FIXED ONLY THIS
    category = str(row["category"]).lower()

    if any(word in category for word in invalid_categories):
        print(f"⏭️ Skipping {name} (not ticket-based)")
        price_ranges.append("NOT_TICKET_BASED")
        continue

    price = get_price_range(name)

    if price:
        print(f"✅ {name} → {price}")
    else:
        print(f"❌ {name} → No price found")

    price_ranges.append(price)

    time.sleep(1)

# -------------------------------
# SAVE RESULTS
# -------------------------------
df["price_range_php"] = price_ranges

# ADD CLEAN COLUMNS (no deletion)
cleaned = df["price_range_php"].apply(clean_price)
df[["price_min", "price_max", "price_unit"]] = pd.DataFrame(cleaned.tolist(), index=df.index)

df.to_csv("final_activities_with_prices.csv", index=False)

print("\n📁 Saved to final_activities_with_prices.csv")
print(df.head())

🔄 STARTING SMART PRICE SCRAPE...



Scraping Prices:   0%|          | 0/513 [00:00<?, ?it/s]

🔗 Checking: https://www.easybook.com/en-ph/tour/destination/philippines/manila/makati/ayala-triangle-gardens
🔗 Checking: https://www.tripadvisor.com/Attraction_Review-g298450-d2453070-Reviews-Ayala_Triangle_Gardens-Makati_Metro_Manila_Luzon.html
🔗 Checking: https://www.hoppler.com.ph/condominiums-for-sale/ayala-triangle-gardens-makati
✅ Ayala Triangle Gardens → FREE


Scraping Prices:   0%|          | 1/513 [00:05<46:33,  5.46s/it]

✅ Ayala Museum → 160-200 (per_ticket)


...

❌ Makati Meditation → No price found


Scraping Prices:  88%|████████▊ | 451/513 [54:53<41:09, 39.83s/it]

❌ Open Mat Makati | Brazilian Jiu Jitsu | Yoga | Cold Plunge | Infrared Sauna | Contrast Therapy → No price found


Scraping Prices:  88%|████████▊ | 452/513 [1:18:39<10:36, 10.44s/it]


KeyboardInterrupt: 